# 33 — Skill Extraction (Rules)
**Goal:** Extract skills from resume text using rule-based matching.

## 1. Building a Skills Database

In [ ]:
# Known skills from ESCO/O*NET taxonomy
SKILLS_DB = {
    "programming": ["Python", "Java", "JavaScript", "TypeScript", "C++", "Go", "Rust", "Scala", "Kotlin", "Ruby", "PHP", "C#", "Swift"],
    "ml_dl": ["TensorFlow", "PyTorch", "scikit-learn", "Keras", "XGBoost", "LightGBM", "JAX"],
    "nlp": ["NLP", "spaCy", "NLTK", "Hugging Face", "Transformers", "BERT", "GPT", "LLM"],
    "data": ["SQL", "Pandas", "NumPy", "Spark", "Hadoop", "Tableau", "Power BI", "Looker"],
    "cloud": ["AWS", "Azure", "GCP", "Docker", "Kubernetes", "Terraform", "Jenkins"],
    "databases": ["PostgreSQL", "MySQL", "MongoDB", "Redis", "Elasticsearch", "Cassandra"],
}

print(f"Total known skills: {sum(len(v) for v in SKILLS_DB.values())}")
for cat, skills in SKILLS_DB.items():
    print(f"  {cat:15s}: {', '.join(skills[:5])}...")

## 2. Regex Skill Matching

In [ ]:
import re

def extract_skills_regex(text, skills_db):
    """Find all known skills in text using word-boundary regex."""
    found = []
    text_lower = text.lower()
    for category, skills in skills_db.items():
        for skill in skills:
            if re.search(r"\\b" + re.escape(skill) + r"\\b", text, re.IGNORECASE):
                found.append({"raw": skill, "category": category, "confidence": 1.0, "method": "exact_match"})
    return found

resume = """Experienced with Python, TensorFlow, and AWS.
Also skilled in NLP, PyTorch, and Kubernetes."""
skills = extract_skills_regex(resume, SKILLS_DB)
for s in skills:
    print(f"  {s['raw']:15s} -> {s['category']:15s} (conf: {s['confidence']})")

## 3. Fuzzy Matching for Typos

In [ ]:
from rapidfuzz import fuzz, process

def extract_skills_fuzzy(text, skills_db, threshold=85):
    """Find skills with fuzzy matching for typos."""
    all_skills = [(cat, s) for cat, skills in skills_db.items() for s in skills]
    words = re.findall(r"\\b[A-Za-z#+]+\\b", text)
    found = set()
    for word in words:
        best_match = process.extractOne(word, [s for _, s in all_skills], scorer=fuzz.ratio)
        if best_match and best_match[1] >= threshold:
            cat = next(c for c, s in all_skills if s == best_match[0])
            found.add((best_match[0], cat, best_match[1]))
    return found

# Test with typo
text = "I know PyTorch, TensrFlow, and Dockr"
for skill, cat, score in extract_skills_fuzzy(text, SKILLS_DB):
    print(f"  '{skill:15s}' -> {cat:15s} (fuzzy: {score}%)")

## Summary: Start with exact regex matching, layer fuzzy matching for typos.